In [90]:
# GAE

In [91]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data
from torch_geometric.utils import train_test_split_edges, negative_sampling
import pandas as pd
import uuid
import time
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score, average_precision_score

from ast import literal_eval
import numpy as np

In [92]:
from torch_geometric.nn import Node2Vec

import warnings
warnings.filterwarnings('ignore')

In [93]:
# Set random seed for reproducibility
random_seed = 42
np.random.seed(random_seed)

# Load datasets
job_descriptions = pd.read_csv('./data/processed/job_descriptions_processed-v3.csv')
resumes = pd.read_csv('./data/processed/general-resume-dataset-processed-v3.csv', converters={'skills': literal_eval})

# Shuffle job_descriptions and select the first N rows
job_descriptions = job_descriptions.sample(frac=1, random_state=random_seed).head(2500)

# Convert 'skills' column to list
job_descriptions['skills'] = job_descriptions['skills'].apply(literal_eval)

In [94]:
# Replace None values in job_title and category with a default value before encoding
job_descriptions['job_title'].fillna('unknown', inplace=True)
resumes['job_title'].fillna('unknown', inplace=True)
resumes['category'].fillna('unknown', inplace=True)

# Ensure job_ids and candidate_ids are correctly assigned
job_descriptions['job_id'] = range(1, len(job_descriptions) + 1)
resumes['candidate_id'] = range(1, len(resumes) + 1)

# Add 'unknown' to the list of all titles and categories to handle unseen labels
all_titles = job_descriptions['job_title'].tolist() + resumes['job_title'].tolist()
all_titles.append('unknown')
all_categories = resumes['category'].tolist()
all_categories.append('unknown')

In [95]:
# Fit the label encoders
le_job_title = LabelEncoder()
le_category = LabelEncoder()
le_job_title.fit(all_titles)
le_category.fit(all_categories)

# Transform the columns
job_descriptions['job_title'] = le_job_title.transform(job_descriptions['job_title'])
resumes['job_title'] = le_job_title.transform(resumes['job_title'])
resumes['category'] = le_category.transform(resumes['category'])

# Encode skills
all_skills = set(skill for skills in job_descriptions['skills'].tolist() + resumes['skills'].tolist() for skill in skills)
le_skills = {skill: i for i, skill in enumerate(all_skills)}

In [96]:
# Create nodes and edges for the graph
nodes = []
edges = []
weights = []
node_features = []

jobs_from_edges = []
candidates_from_edges = []
jobs_and_candidates_from_edges = []

skill_weight_multiplier = 3  # Weight for skill overlap
title_weight = 5  # Weight for job title match

# Add job nodes
for i, row in job_descriptions.iterrows():
    nodes.append(row['job_id'])
    skills_vector = [0] * len(le_skills)
    if row['skills']:  # Check if skills are not empty
        for skill in row['skills']:
            skills_vector[le_skills[skill]] = 1
    node_features.append([row['job_title']] + skills_vector)

# Add resume nodes, using 'category' instead of 'job_title'
for i, row in resumes.iterrows():
    nodes.append(row['candidate_id'] + len(job_descriptions))
    skills_vector = [0] * len(le_skills)
    if row['skills']:  # Check if skills are not empty
        for skill in row['skills']:
            skills_vector[le_skills[skill]] = 1
    node_features.append([row['job_title']] + skills_vector)

In [97]:
# Convert job_skills and resume_skills to sets once
job_descriptions['skills'] = job_descriptions['skills'].apply(set)
resumes['skills'] = resumes['skills'].apply(set)

# Create dictionaries for quick lookup
job_skills_dict = job_descriptions.set_index('job_id')['skills'].to_dict()
resume_skills_dict = resumes.set_index('candidate_id')['skills'].to_dict()
job_titles_dict = job_descriptions.set_index('job_id')['job_title'].to_dict()
resume_titles_dict = resumes.set_index('candidate_id')['job_title'].to_dict()

edges = []
weights = []
jobs_and_candidates_from_edges = []
jobs_from_edges = []
candidates_from_edges = []

for job_id, job_skills in job_skills_dict.items():
    for candidate_id, resume_skills in resume_skills_dict.items():
        overlap = len(job_skills.intersection(resume_skills))
        combined_weight = overlap * skill_weight_multiplier

        if job_titles_dict[job_id] == resume_titles_dict[candidate_id]:
            combined_weight += title_weight  # Add weight for job title match

        if combined_weight > 0:
            edges.append((job_id, candidate_id + len(job_descriptions)))
            weights.append(combined_weight)
            jobs_and_candidates_from_edges.append((job_id, candidate_id))
            jobs_from_edges.append(job_id)
            candidates_from_edges.append(candidate_id)

nodes_length = len(nodes)

In [98]:
edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
edge_weight = torch.tensor(weights, dtype=torch.float)
edge_index = edge_index.clamp(0, nodes_length - 1)

In [99]:
# torch.save(edge_index, './data/features/edge_index-2500-a02.pt')
# torch.save(edge_weight, './data/features/edge_weight-2500-a02.pt')
# with open('./data/features/nodes_len-2500-a02.txt', 'w') as file:
#     file.write(str(len(nodes)))

In [100]:
# edge_index = torch.load('./data/features/edge_index-2500-a02.pt')
# edge_weight = torch.load('./data/features/edge_weight-2500-a02.pt')
# nodes_length = 0
# with open('./data/features/nodes_len-2500-a02.txt', 'r') as file:
#     nodes_length = np.int64(file.read())

In [101]:
# Convert node features to tensor
x = torch.tensor(node_features, dtype=torch.float)

In [102]:
# torch.save(x, './data/features/x-2500-a02.pt')

In [103]:
# x = torch.load('./data/features/x-2500-a02.pt')

In [104]:
# Create PyTorch Geometric data object
data = Data(x=x, edge_index=edge_index, edge_weight=edge_weight)

# Ensure edge indices are within range
# data.num_nodes = nodes_length

original_edge_index = data.edge_index.clone()
# original_edge_weight = data.edge_weight.clone()

# Splitting edges for training/validation
data = train_test_split_edges(data)

In [105]:
# Create a dictionary to map edge indices to their weights
edge_weight_dict = {tuple(edge_index[:, i].tolist()): edge_weight[i].item() for i in range(edge_index.size(1))}

def get_edge_weights(edge_index, edge_weight_dict):
    weights = []
    for i in range(edge_index.size(1)):
        edge = tuple(edge_index[:, i].tolist())
        weight = edge_weight_dict.get(edge, 0)  # Default to 0 if edge not found
        weights.append(weight)
    return torch.tensor(weights, dtype=torch.float)

In [106]:
train_edge_weights = get_edge_weights(data.train_pos_edge_index, edge_weight_dict)
test_edge_weights = get_edge_weights(data.test_pos_edge_index, edge_weight_dict)
val_edge_weights = get_edge_weights(data.val_pos_edge_index, edge_weight_dict)

In [107]:
data.train_pos_edge_weight = train_edge_weights
data.test_pos_edge_weight = test_edge_weights
data.val_pos_edge_weight = val_edge_weights

# Manually create negative edges for training
neg_edge_index_train = negative_sampling(
    edge_index=data.train_pos_edge_index,
    num_nodes=data.num_nodes,
    num_neg_samples=data.train_pos_edge_index.size(1),
)
data.train_neg_edge_index = neg_edge_index_train

# Assign zero weights to negative edges for training
neg_train_edge_weights = torch.zeros(neg_edge_index_train.size(1), dtype=torch.float)

# Manually create negative edges for testing
neg_edge_index_test = negative_sampling(
    edge_index=data.test_pos_edge_index,
    num_nodes=data.num_nodes,
    num_neg_samples=data.test_pos_edge_index.size(1),
)
data.test_neg_edge_index = neg_edge_index_test

# Assign zero weights to negative edges for testing
neg_test_edge_weights = torch.zeros(neg_edge_index_test.size(1), dtype=torch.float)

# Combine positive and negative edge weights for training
data.train_neg_edge_weight = neg_train_edge_weights

# Combine positive and negative edge weights for testing
data.test_neg_edge_weight = neg_test_edge_weights

# Ensure edge_index tensors are of integer type
data.train_pos_edge_index = data.train_pos_edge_index.long()
data.test_pos_edge_index = data.test_pos_edge_index.long()
data.train_neg_edge_index = data.train_neg_edge_index.long()
data.test_neg_edge_index = data.test_neg_edge_index.long()

In [108]:
# Define number of nodes
num_nodes = data.num_nodes

# Node2Vec parameters
embedding_dim = 64
walk_length = 20
context_size = 10
walks_per_node = 10
batch_size = 128
lr = 0.01
num_epochs = 11

In [109]:
# # Initialize Node2Vec
# node2vec = Node2Vec(
#     edge_index=original_edge_index,
#     embedding_dim=embedding_dim,
#     walk_length=walk_length,
#     context_size=context_size,
#     walks_per_node=walks_per_node,
#     num_negative_samples=1,
#     p=1,
#     q=1,
#     sparse=True,
#     num_nodes=num_nodes  # Specify number of nodes
# )
#
# # Move to the appropriate device (CPU/GPU)
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# node2vec = node2vec.to(device)

In [110]:
# # Optimizer for Node2Vec
# optimizer = torch.optim.SparseAdam(node2vec.parameters(), lr=lr)

In [111]:
# # Training loop for Node2Vec
# def train_node2vec(num_epochs):
#     node2vec.train()
#     for epoch in range(num_epochs):
#         total_loss = 0
#         loader = node2vec.loader(batch_size=batch_size, shuffle=True, num_workers=0)  # Set num_workers=0
#         for i, (pos_rw, neg_rw) in enumerate(loader):
#             optimizer.zero_grad()
#             loss = node2vec.loss(pos_rw.to(device), neg_rw.to(device))
#             loss.backward()
#             optimizer.step()
#             total_loss += loss.item()
#             if epoch % 10 == 0:
#                 print(f'Epoch {epoch + 1}, Iteration {i}, Loss: {total_loss / 10}')
#                 total_loss = 0

In [112]:
# # Train Node2Vec model
# train_node2vec(num_epochs)

In [113]:
# # Extract embeddings
# node_embeddings = node2vec.embedding.weight.data.cpu().numpy()
#
# # Update node features with embeddings
# data.x = torch.tensor(node_embeddings, dtype=torch.float)

In [114]:
class GAE(torch.nn.Module):
    def __init__(self, in_channels, out_channels):
        super(GAE, self).__init__()
        self.conv1 = GCNConv(in_channels, 2 * out_channels)
        self.conv2 = GCNConv(2 * out_channels, out_channels)

    def encode(self, x, edge_index, edge_weight):
        x = F.relu(self.conv1(x, edge_index, edge_weight))
        return self.conv2(x, edge_index, edge_weight)

    def decode(self, z, pos_edge_index, neg_edge_index):
        pos_pred = (z[pos_edge_index[0].long()] * z[pos_edge_index[1].long()]).sum(dim=1)
        neg_pred = (z[neg_edge_index[0].long()] * z[neg_edge_index[1].long()]).sum(dim=1)
        return pos_pred, neg_pred

    def forward(self, data):
        z = self.encode(data.x, data.train_pos_edge_index, data.train_pos_edge_weight)
        return z

In [115]:
# Initialize and train GAE model as before
model = GAE(data.num_node_features, 32)  # Adjust dimensions as needed
gae_optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = torch.nn.BCEWithLogitsLoss()

In [116]:
def train(data):
    model.train()
    gae_optimizer.zero_grad()
    z = model.encode(data.x, data.train_pos_edge_index, data.train_pos_edge_weight)  # Pass train_pos_edge_weight
    pos_pred, neg_pred = model.decode(z, data.train_pos_edge_index, data.train_neg_edge_index)
    pos_loss = loss_fn(pos_pred, torch.ones_like(pos_pred))
    neg_loss = loss_fn(neg_pred, torch.zeros_like(neg_pred))
    loss = pos_loss + neg_loss
    if torch.isnan(loss) or torch.isinf(loss):
        print("Warning: NaN or Inf loss detected")
        return float('inf')
    loss.backward()
    gae_optimizer.step()
    return loss.item()

In [117]:
for epoch in range(450):
    loss = train(data)
    if loss == float('inf'):
        break
    if epoch % 10 == 0:
        print(f'Epoch {epoch}, Loss: {loss}')

Epoch 0, Loss: 5000.6396484375
Epoch 10, Loss: 52.79936599731445
Epoch 20, Loss: 1.3871383666992188
Epoch 30, Loss: 1.3878051042556763
Epoch 40, Loss: 1.3877718448638916
Epoch 50, Loss: 1.3870089054107666
Epoch 60, Loss: 1.3855807781219482
Epoch 70, Loss: 1.3836774826049805
Epoch 80, Loss: 1.3812227249145508
Epoch 90, Loss: 1.378005027770996
Epoch 100, Loss: 1.373734474182129
Epoch 110, Loss: 1.368091344833374
Epoch 120, Loss: 1.360830545425415
Epoch 130, Loss: 1.3519377708435059
Epoch 140, Loss: 1.3402024507522583
Epoch 150, Loss: 1.3266315460205078
Epoch 160, Loss: 1.3128986358642578
Epoch 170, Loss: 1.2995598316192627
Epoch 180, Loss: 1.2872254848480225
Epoch 190, Loss: 1.2766352891921997
Epoch 200, Loss: 1.2681303024291992
Epoch 210, Loss: 1.2614691257476807
Epoch 220, Loss: 1.2554130554199219
Epoch 230, Loss: 1.2497730255126953
Epoch 240, Loss: 1.2453327178955078
Epoch 250, Loss: 1.2417877912521362
Epoch 260, Loss: 1.238935112953186
Epoch 270, Loss: 1.2366068363189697
Epoch 280, L

In [118]:
def precision_at_k(y_true, y_pred, k):
    idx = np.argsort(y_pred)[::-1][:k]
    y_pred_binary = np.zeros_like(y_pred)
    y_pred_binary[idx] = 1
    tp = np.sum(y_true * y_pred_binary)
    precision = tp / k
    return precision

def recall_at_k(y_true, y_pred, k):
    idx = np.argsort(y_pred)[::-1][:k]
    y_pred_binary = np.zeros_like(y_pred)
    y_pred_binary[idx] = 1
    tp = np.sum(y_true * y_pred_binary)
    recall = tp / np.sum(y_true)
    return recall

def average_precision(y_true, y_pred):
    idx = np.argsort(y_pred)[::-1]
    y_true_sorted = y_true[idx]
    tp = np.cumsum(y_true_sorted)
    precision = tp / (np.arange(len(y_true_sorted)) + 1)
    avg_precision = np.sum(precision * y_true_sorted) / np.sum(y_true_sorted)
    return avg_precision

def mean_average_precision(y_true, y_pred):
    return np.mean([average_precision(y_t, y_p) for y_t, y_p in zip(y_true, y_pred)])

def dcg_score(y_true, y_pred, k):
    order = np.argsort(y_pred)[::-1]
    y_true_sorted = np.take(y_true, order[:k])
    gain = 2 ** y_true_sorted - 1
    discounts = np.log2(np.arange(len(y_true_sorted)) + 2)
    return np.sum(gain / discounts)

def ndcg_score(y_true, y_pred, k):
    best = dcg_score(y_true, y_true, k)
    actual = dcg_score(y_true, y_pred, k)
    return actual / best

In [119]:
def evaluate_model(data, model, k):
    model.eval()
    with torch.no_grad():
        z = model.encode(data.x, data.val_pos_edge_index, data.val_pos_edge_weight)
        pos_pred = torch.sigmoid((z[data.val_pos_edge_index[0].long()] * z[data.val_pos_edge_index[1].long()]).sum(dim=1)).cpu().numpy()
        neg_pred = torch.sigmoid((z[data.val_neg_edge_index[0].long()] * z[data.val_neg_edge_index[1].long()]).sum(dim=1)).cpu().numpy()

    y_true = np.concatenate([np.ones(pos_pred.shape[0]), np.zeros(neg_pred.shape[0])])
    y_pred = np.concatenate([pos_pred, neg_pred])

    auc_roc = roc_auc_score(y_true, y_pred)
    ap = average_precision_score(y_true, y_pred)

    precision = precision_at_k(y_true, y_pred, k)
    recall = recall_at_k(y_true, y_pred, k)
    map_score = mean_average_precision([y_true], [y_pred])
    ndcg = ndcg_score(y_true, y_pred, k)

    return auc_roc, ap, precision, recall, map_score, ndcg

In [120]:
k=10

In [121]:
# Example usage
auc_roc, ap, k_prec, recall, map_score, ndcg = evaluate_model(data, model, k)
print(f"AUC-ROC: {auc_roc:.4f}, AP: {ap:.4f}")
print(f"Precision@{k}: {k_prec:.4f}")
print(f"Recall@{k}: {recall:.4f}")
print(f"MAP: {map_score:.4f}")
print(f"NDCG@{k}: {ndcg:.4f}")

AUC-ROC: 0.6366, AP: 0.6000
Precision@10: 1.0000
Recall@10: 0.0034
MAP: 0.8578
NDCG@10: 1.0000


In [122]:
def predict_best_candidates(job_descriptions, resumes, z, k=1):
    job_ids = job_descriptions['job_id'].values
    candidate_ids = resumes['candidate_id'].values + len(job_descriptions)

    job_indices = job_ids - 1
    candidate_indices = candidate_ids - 1

    job_embeddings = z[job_indices]
    candidate_embeddings = z[candidate_indices]

    # Calculate scores using matrix multiplication
    scores = torch.sigmoid(torch.matmul(job_embeddings, candidate_embeddings.T)).cpu().numpy()

    predictions = []
    for i, job_id in enumerate(job_ids):
        best_match_indices = scores[i].argsort()[::-1][:k]
        for idx in best_match_indices:
            candidate_id = resumes.iloc[idx]['candidate_id']
            candidate_job_title = resumes.iloc[idx]['job_title']
            category = resumes.iloc[idx]['category']
            skills = resumes.iloc[idx]['skills']
            job_title = job_descriptions.iloc[i]['job_title']
            job_skills = job_descriptions.iloc[i]['skills']
            score = scores[i][idx]

            match_percentage = score * 100  # Assuming the score is between 0 and 1
            predictions.append({
                "Job ID": job_id,
                "Job Title": le_job_title.inverse_transform([job_title])[0],
                "Candidate ID": candidate_id,
                "Candidate Job Title": le_job_title.inverse_transform([candidate_job_title])[0],
                "Candidate Category": le_category.inverse_transform([category])[0],
                "Match Percentage": match_percentage,
                "Mutual Skills": set(job_skills).intersection(set(skills)),
                "Job Skills": job_skills,
                "Candidate Skills": skills
            })

    predictions_df = pd.DataFrame(predictions)
    return predictions_df

In [123]:
# Example usage
with torch.no_grad():
    z = model.encode(data.x, data.test_pos_edge_index, data.test_pos_edge_weight)

In [124]:
jobs_to_predict = job_descriptions[job_descriptions['job_title'].isin(resumes['job_title'])].sample(frac=1, random_state=random_seed)

In [125]:
predictions_df = predict_best_candidates(job_descriptions, resumes, z)
predictions_df = predictions_df[predictions_df['Mutual Skills'].map(len) != 0]
ind = predictions_df['Mutual Skills'].map(len).sort_values(ascending=False).index
predictions_df = predictions_df.reindex(ind)

In [126]:
predictions_df.head(2500)

,Job ID,Job Title,Candidate ID,Candidate Job Title,Candidate Category,Match Percentage,Mutual Skills,Job Skills,Candidate Skills
1002,1003,operations manager,235,director of finance,finance,96.994331,"{strategic planning, process improvement, fina...","{management process, process improvement, team...","{balance sheet, cost accounting, goal set, phy..."
2041,2042,operations manager,235,director of finance,finance,96.994331,"{strategic planning, process improvement, fina...","{management process, process improvement, team...","{balance sheet, cost accounting, goal set, phy..."
148,149,operations manager,235,director of finance,finance,96.994331,"{strategic planning, process improvement, fina...","{management process, process improvement, team...","{balance sheet, cost accounting, goal set, phy..."
1003,1004,customer service manager,235,director of finance,finance,96.994331,"{quality control, process improvement}","{quality assurance, quality control, process i...","{balance sheet, cost accounting, goal set, phy..."
1137,1138,qa engineer,235,director of finance,finance,96.994331,"{quality control, process improvement}","{process improvement, team leadership, quality...","{balance sheet, cost accounting, goal set, phy..."
...,...,...,...,...,...,...,...,...,...
818,819,purchasing agent,235,director of finance,finance,96.994331,{inventory control},"{supply chain management, supply chain, demand...","{balance sheet, cost accounting, goal set, phy..."
880,881,investment analyst,235,director of finance,finance,96.994331,{financial analysis},"{portfolio optimization, asset allocation, inv...","{balance sheet, cost accounting, goal set, phy..."
882,883,office manager,235,director of finance,finance,96.994331,{process improvement},"{office management, management process, financ...","{balance sheet, cost accounting, goal set, phy..."
898,899,financial analyst,235,director of finance,finance,96.994331,{financial analysis},"{portfolio management, financial analysis}","{balance sheet, cost accounting, goal set, phy..."


In [127]:
# torch.save(model.state_dict(), f"./models/gea-recommendation-system-25k-{auc_roc:.2f}-acc-{uuid.uuid4()}-{time.strftime('%Y%m%d-%H%M%S')}-v4.pth")

In [128]:
# torch.save(model.state_dict(), f"./models/gea-recommendation-system-v4-a02.pth")

In [136]:
def predict_for_new_job_description(new_job_desc, job_descriptions, resumes, model, le_job_title, le_skills, le_category, skill_weight_multiplier=3, title_weight=5, k=1):
    """
    Predict the best candidate(s) for a new job description with handling for unseen labels.

    Parameters:
    - new_job_desc (dict): New job description with 'job_title' and 'skills'.
    - job_descriptions (pd.DataFrame): Original job descriptions.
    - resumes (pd.DataFrame): Candidate resumes.
    - model (GAE): Trained GAE model.
    - le_job_title (LabelEncoder): Encoder for job titles.
    - le_skills (dict): Dictionary mapping skills to indices.
    - le_category (LabelEncoder): Encoder for categories.
    - skill_weight_multiplier (int): Weight for skill overlap.
    - title_weight (int): Weight for title match.
    - k (int): Number of top candidates to return.

    Returns:
    - pd.DataFrame: DataFrame with top-k candidates and match details.
    """
    # Handle unseen job title
    try:
        job_title = le_job_title.transform([new_job_desc['job_title']])[0]
    except ValueError:
        job_title = le_job_title.transform(['unknown'])[0]  # Fallback to 'unknown'

    # Encode skills with fallback for unseen skills
    skills_vector = [0] * len(le_skills)
    for skill in new_job_desc['skills']:
        if skill in le_skills:
            skills_vector[le_skills[skill]] = 1  # Encode known skills
        else:
            # Log a warning for unseen skills (optional)
            print(f"Warning: Unseen skill '{skill}' not found in encoder.")

    # Create the raw feature vector for the new job
    new_job_features = torch.tensor([job_title] + skills_vector, dtype=torch.float).unsqueeze(0)

    # Update the graph with the new job node
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    original_x = data.x
    original_edge_index = data.edge_index

    # Append the new job node to the feature matrix
    updated_x = torch.cat([original_x, new_job_features.to(device)], dim=0)

    # Add edges between the new job node and candidate nodes
    new_node_index = updated_x.size(0) - 1  # Index of the new job node
    candidate_indices = resumes['candidate_id'].values + len(job_descriptions) - 1
    new_edges = torch.tensor(
        [[new_node_index] * len(candidate_indices), candidate_indices],
        dtype=torch.long,
        device=device
    )

    # Handle missing original_edge_index
    if original_edge_index is None:
        updated_edge_index = new_edges
    else:
        updated_edge_index = torch.cat([original_edge_index, new_edges], dim=1)

    # Compute embeddings using the updated graph
    model.eval()
    with torch.no_grad():
        z = model.encode(updated_x, updated_edge_index, None)

    # Get the embedding for the new job node
    new_job_embedding = z[new_node_index].unsqueeze(0)

    # Get embeddings for candidates
    candidate_embeddings = z[candidate_indices]

    # Calculate similarity scores (dot product) between the new job and candidates
    scores = torch.sigmoid(torch.matmul(new_job_embedding, candidate_embeddings.T)).cpu().numpy()

    # Rank candidates by scores and select the top-k
    top_k_indices = scores[0].argsort()[::-1][:k]
    predictions = []
    for idx in top_k_indices:
        candidate_id = resumes.iloc[idx]['candidate_id']
        candidate_job_title = resumes.iloc[idx]['job_title']
        category = resumes.iloc[idx]['category']
        candidate_skills = resumes.iloc[idx]['skills']
        mutual_skills = set(new_job_desc['skills']).intersection(candidate_skills)
        match_percentage = scores[0][idx] * 100  # Convert score to percentage

        predictions.append({
            "Job Title": new_job_desc['job_title'],
            "Job Skills": new_job_desc['skills'],
            "Candidate ID": candidate_id,
            "Candidate Job Title": le_job_title.inverse_transform([candidate_job_title])[0],
            "Candidate Category": le_category.inverse_transform([category])[0],
            "Candidate Skills": candidate_skills,
            "Mutual Skills": list(mutual_skills),
            "Match Percentage": match_percentage,
        })

    predictions_df = pd.DataFrame(predictions)
    return predictions_df

In [143]:
new_job_desc = {
    "job_title": "director of finance",  # This might be unseen
    "skills": ["strategic planning", "process improvement", "financial analysis"]
}

predicted_candidates_df = predict_for_new_job_description(
    new_job_desc=new_job_desc,
    job_descriptions=job_descriptions,
    resumes=resumes,
    model=model,
    le_job_title=le_job_title,
    le_skills=le_skills,
    le_category=le_category,
    k=10  # Number of top candidates to return
)

predicted_candidates_df.head()

,Job Title,Job Skills,Candidate ID,Candidate Job Title,Candidate Category,Candidate Skills,Mutual Skills,Match Percentage
0,director of finance,"[strategic planning, process improvement, fina...",250,accountant,accountant,"{mba, petty cash, financial statement, payroll...",[],53.37431
1,director of finance,"[strategic planning, process improvement, fina...",79,kids club attendant,fitness,"{microsoft word, customer service, africana st...",[],53.37431
2,director of finance,"[strategic planning, process improvement, fina...",92,accountant,accountant,"{management accounting, cash flow statement, b...",[process improvement],53.37431
3,director of finance,"[strategic planning, process improvement, fina...",91,biomedical engineering technician ii,engineering,"{system support, preventive maintenance, self ...",[],53.37431
4,director of finance,"[strategic planning, process improvement, fina...",90,construction manager,construction,"{project coordination, team leadership, comple...",[strategic planning],53.37431


In [144]:
# Save the GAE model
model_path = "gae_model.pth"
torch.save(model.state_dict(), model_path)
print(f"Model saved to {model_path}")

Model saved to gae_model.pth


In [145]:
import pickle

# Save the graph data object
data_path = "graph_data.pkl"
with open(data_path, "wb") as f:
    pickle.dump(data, f)
print(f"Graph data saved to {data_path}")

# Save the encoders
encoders_path = "encoders.pkl"
encoders = {
    "le_job_title": le_job_title,
    "le_skills": le_skills,
    "le_category": le_category
}
with open(encoders_path, "wb") as f:
    pickle.dump(encoders, f)
print(f"Encoders saved to {encoders_path}")

Graph data saved to graph_data.pkl
Encoders saved to encoders.pkl


In [146]:
# Load the GAE model
loaded_model = GAE(data.num_node_features, 32)  # Ensure the architecture matches
loaded_model.load_state_dict(torch.load(model_path))
loaded_model.eval()
print("Model loaded successfully.")

# Load the graph data object
with open(data_path, "rb") as f:
    loaded_data = pickle.load(f)
print("Graph data loaded successfully.")

# Load the encoders
with open(encoders_path, "rb") as f:
    loaded_encoders = pickle.load(f)
    le_job_title = loaded_encoders["le_job_title"]
    le_skills = loaded_encoders["le_skills"]
    le_category = loaded_encoders["le_category"]
print("Encoders loaded successfully.")

Model loaded successfully.
Graph data loaded successfully.
Encoders loaded successfully.


In [147]:
# Example: Predict candidates for a new job description
new_job_desc = {
    "job_title": "Data Scientist",
    "skills": ["Python", "Machine Learning", "SQL"]
}

# Predict top candidates using the loaded model and data
predicted_candidates_df = predict_for_new_job_description(
    new_job_desc=new_job_desc,
    job_descriptions=job_descriptions,  # Reloaded if necessary
    resumes=resumes,  # Reloaded if necessary
    model=loaded_model,
    le_job_title=le_job_title,
    le_skills=le_skills,
    le_category=le_category,
    k=3  # Number of top candidates to return
)

predicted_candidates_df.head()

,Job Title,Job Skills,Candidate ID,Candidate Job Title,Candidate Category,Candidate Skills,Mutual Skills,Match Percentage
0,Data Scientist,"[Python, Machine Learning, SQL]",250,accountant,accountant,"{mba, petty cash, financial statement, payroll...",[],53.37431
1,Data Scientist,"[Python, Machine Learning, SQL]",79,kids club attendant,fitness,"{microsoft word, customer service, africana st...",[],53.37431
2,Data Scientist,"[Python, Machine Learning, SQL]",92,accountant,accountant,"{management accounting, cash flow statement, b...",[],53.37431
